In [1]:
import pandas as pd
import psycopg2
from getpass import getpass
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

conn = psycopg2.connect(host="localhost", port=5432, dbname="olist",
                        user="postgres", password=getpass("Пароль postgres: "))

# 3.4.3 / 3.4.4: повторные покупки в зависимости от опоздания ПЕРВОГО заказа
QUERY = """
WITH ord AS (
    SELECT c.customer_unique_id,
           o.order_purchase_timestamp,
           o.order_delivered_customer_date,
           o.order_estimated_delivery_date,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                              ORDER BY o.order_purchase_timestamp) AS rn,
           COUNT(*) OVER (PARTITION BY c.customer_unique_id)       AS n_orders
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
      AND o.order_delivered_customer_date IS NOT NULL
)
SELECT CASE WHEN order_delivered_customer_date::date > order_estimated_delivery_date::date
            THEN 'late' ELSE 'on_time' END              AS first_delivery,
       COUNT(*)                                         AS customers,
       COUNT(*) FILTER (WHERE n_orders > 1)             AS repeaters
FROM ord
WHERE rn = 1
  AND order_purchase_timestamp < %(cutoff)s
GROUP BY first_delivery
ORDER BY first_delivery;
"""

for cutoff, label in [("2018-09-01", "3.4.3 — все покупатели"),
                      ("2018-03-01", "3.4.4 — первая покупка до 2018-03-01")]:
    df = pd.read_sql(QUERY, conn, params={"cutoff": cutoff})
    df["repeat_pct"] = (100 * df["repeaters"] / df["customers"]).round(2)
    z, p = proportions_ztest(count=df["repeaters"], nobs=df["customers"])
    print(f"\n{label}")
    print(df.to_string(index=False))
    print(f"z = {z:.2f}, p = {p:.4f}")

conn.close()

Пароль postgres:  ········



3.4.3 — все покупатели
first_delivery  customers  repeaters  repeat_pct
          late       6356        164        2.58
       on_time      86740       2625        3.03
z = -2.01, p = 0.0441

3.4.4 — первая покупка до 2018-03-01
first_delivery  customers  repeaters  repeat_pct
          late       3675        115        3.13
       on_time      51590       2082        4.04
z = -2.72, p = 0.0066


## Вывод
Покупатели, у которых первый заказ доставили позже обещанной даты, возвращаются реже:
- все покупатели: 2,58% против 3,03%, z = −2,01, p = 0,044;
- только с первой покупкой до 2018-03-01 (не меньше 6 месяцев на возврат): 3,13% против 4,04% (−23%), z = −2,72, p = 0,007.

Разница сохраняется и усиливается, когда у всех было время вернуться, — это не артефакт «не успел вернуться».
Ограничение: это наблюдательные данные, связь ≠ причинность (возможны общие причины — продавец, категория, регион).
Для причинного вывода — A/B-тест обещанной даты доставки (дизайн — в README).